In [191]:
import pandas as pd
import regex as re

In [192]:
data = pd.read_csv("data/translation_data_full_duplicates.csv")


In [193]:
data

,teaviku_laad,sihtrühm,pealkiri,autor,originaal_pealkiri,Žanr,tõlkija,lähtekeel,lähtekirjandus,toimetaja,...,ilmumisaasta,füüsiline_kirjeldus,trükikordus,sarjaandmed,arvustatava_tõlkeraamatu_autor,väljaanne,number_leheküljed,sisu,märkused,id
0,Tõlkearvustus,Täiskasvanute ilukirjandus,Natanael. Kulturhistorialine roman. A. Liepe j...,π,NaN,romaanid,"Meomuttel, Jüri 1868-1948",saksa,saksa,NaN,...,1896,NaN,NaN,NaN,"Liepe, Albert",<p>Olevik</p>,"13. veebr., nr 7, lk 161-162",NaN,Idna Illiv (pseud.) = Jüri Meomuttel,71113
1,Tõlkearvustus,Täiskasvanute ilukirjandus,Talu sulane Tõnis. Ühe Schweitsi jutu järele j...,π,NaN,jutud,teadmata,saksa,šveitsi,NaN,...,1897,NaN,NaN,NaN,"Stael von Holstein, Lucia 1857-1929",<p>Olevik</p>,"28. okt., nr 43, lk 961",NaN,NaN,53196
2,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,Miss Portsia,"Zunk, Paul",teadmata,jutud,"Mällow, J.",teadmata,teadmata,NaN,...,1894,NaN,NaN,NaN,NaN,<p>Eesti Postimees : Lisaleht</p>,nr 21-22,NaN,Väljaandes: Paul Zunk'i uudisjutukese järele J...,63698
3,Kogumikus ilmunud tõlge,Täiskasvanute ilukirjandus,Kodu,"Žukovski, Vassili 1783-1852",NaN,luuletused,"Leppik, Jaan 1861-1943",vene,vene,NaN,...,1889,39 lk. 13 cm,NaN,NaN,NaN,NaN,NaN,<p>Ilmunud kogumikus: Eestistatud laulud 2.</p>,Väljaandes: Wene keelest [autor märkimata]; Ee...,58892
4,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,Kolm õde,"Žukovski, Vassili 1783-1852",teadmata,jutud,G. Kewade,saksa,vene,NaN,...,1886,NaN,NaN,NaN,NaN,<p>Meelelahutaja</p>,"31. dets., nr. 52, lk. 409",NaN,Väljaandes: Shukowsky järele: G. Kewade,55755
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52036,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,Peremeheta kohver,"Adamov, Arkadi 1920-1991",Чемодан без хозяина,jutustused,"Jürna, Juhan-Kaspar 1931-1982",vene,vene,NaN,...,19721973,NaN,NaN,NaN,NaN,<p>Noorte Hääl</p>,"30. nov.; 1.-3., 5., 7.-8., 10., 12., 13.-17.,...",NaN,NaN,82166
52037,Perioodikas ilmunud tõlge,Laste- ja noortekirjandus,Nõiatulp : ulmejutt,"Abramov, Sergei 1944-2024",teadmata,jutud,"Maksing, Meta",vene,vene,NaN,...,19801981,NaN,NaN,NaN,NaN,<p>Säde</p>,"12., 16., 26. juuli; 2., 6., 9., 16., 23., 27....",NaN,NaN,78190
52038,Tõlkearvustus,Täiskasvanute ilukirjandus,Zsigmond Móricz: Mudakuld. Romaan. Ungari keel...,A. K.,Sárarany,romaanid,"Murakin, Ants 1892-1975",ungari,ungari,NaN,...,12929,NaN,NaN,Looduse kroonine romaan nr 1,"Móricz, Zsigmond 1879-1942",<p>Päevaleht</p>,"3. veebr., nr 33, lk 8",NaN,NaN,52312
52039,Tõlkeraamat,Täiskasvanute ilukirjandus,Irmgard : hästi põnew romaan,NaN,teadmata,romaanid,"Neumann, Mihkel 1866-1941",teadmata/undetermined,teadmata/undetermined,NaN,...,19231924,23 annet (733 lk.),NaN,NaN,NaN,NaN,NaN,NaN,Kaanel pealkirja täiendandmed: Ülipõnew romaan...,45109


# Goal: make one big table such that every row of translations is represented, and then all the others as well

## Explode all relevant columns at semicolons

In [194]:
data.columns

Index(['teaviku_laad', 'sihtrühm', 'pealkiri', 'autor', 'originaal_pealkiri',
       'Žanr', 'tõlkija', 'lähtekeel', 'lähtekirjandus', 'toimetaja',
       'ees__järelsõna_autor', 'ilmumiskoht', 'kirjastaja', 'ilmumisaasta',
       'füüsiline_kirjeldus', 'trükikordus', 'sarjaandmed',
       'arvustatava_tõlkeraamatu_autor', 'väljaanne', 'number_leheküljed',
       'sisu', 'märkused', 'id'],
      dtype='str')

In [198]:
data.trükikordus.unique()

<StringArray>
[                                                                                         nan,
                                                                                '2. tr. 1865',
                                                                                     '2. tr.',
                                                                    '2. tr. [esmatrükk 1885]',
                                                                                 '2. tr 1885',
                                                                                     '3. tr.',
                                                                                 '2 tr. 1867',
                                                                    '2. tr. [esmatrükk 1890]',
                                                                                   '2. trükk',
                                                            '3. trükk, esimesed 1841 ja 1842',
                                    

In [199]:
semicolon_splittable = ['author',
       'genre', 'translator', 'source_language',
       'source_literature', 'editor', 'fore_afterword_author',
       'publication_location', 'publisher', 'series',
       'author_of_translated_book_under_review_to_be_removed']
semicolon_splittable = ['autor', 
       'Žanr', 'tõlkija', 'lähtekeel', 'lähtekirjandus', 'toimetaja',
       'ees__järelsõna_autor', 'ilmumiskoht', 'kirjastaja', 'sarjaandmed',
       'arvustatava_tõlkeraamatu_autor']
for label in semicolon_splittable:
    data[label] = data[label].str.split(';')
    data = data.explode(label)
    data[label] = data[label].str.strip()

In [200]:
data = data.reset_index(drop=True)

In [201]:
data.columns

Index(['teaviku_laad', 'sihtrühm', 'pealkiri', 'autor', 'originaal_pealkiri',
       'Žanr', 'tõlkija', 'lähtekeel', 'lähtekirjandus', 'toimetaja',
       'ees__järelsõna_autor', 'ilmumiskoht', 'kirjastaja', 'ilmumisaasta',
       'füüsiline_kirjeldus', 'trükikordus', 'sarjaandmed',
       'arvustatava_tõlkeraamatu_autor', 'väljaanne', 'number_leheküljed',
       'sisu', 'märkused', 'id'],
      dtype='str')

In [202]:
data.to_csv("data/exploded_data.csv", index=False)

In [176]:
correct_colindex = ["author", "translator", "editor", "fore_afterword_author", 
                    "title", "title_original", "genre", "source_language", "source_literature",
                   "publisher", "publication_location", "publication_year", "physical_description",
                   "series", "publication_type", "target_audience", "edition", "n_pages", "content",
                   "issue", "notes"]

In [177]:
for col in data.columns:
    if col not in correct_colindex:
        print(f"{col} is not in data.columns")

author_of_translated_book_under_review_to_be_removed is not in data.columns
id is not in data.columns


In [178]:
data = data.drop(["id", "author_of_translated_book_under_review_to_be_removed"], axis=1)

In [179]:
correct_colindex

['author',
 'translator',
 'editor',
 'fore_afterword_author',
 'title',
 'title_original',
 'genre',
 'source_language',
 'source_literature',
 'publisher',
 'publication_location',
 'publication_year',
 'physical_description',
 'series',
 'publication_type',
 'target_audience',
 'edition',
 'n_pages',
 'content',
 'issue',
 'notes']

In [180]:
data = data.loc[:, correct_colindex]

In [181]:
data

,author,translator,editor,fore_afterword_author,title,title_original,genre,source_language,source_literature,publisher,...,publication_year,physical_description,series,publication_type,target_audience,edition,n_pages,content,issue,notes
0,π,"Meomuttel, Jüri 1868-1948",NaN,NaN,Natanael. Kulturhistorialine roman. A. Liepe j...,NaN,romaanid,saksa,saksa,NaN,...,1896.0,NaN,NaN,Tõlkearvustus,Täiskasvanute ilukirjandus,NaN,"13. veebr., nr 7, lk 161-162",NaN,<p>Olevik</p>,Idna Illiv (pseud.) = Jüri Meomuttel
1,π,teadmata,NaN,NaN,Talu sulane Tõnis. Ühe Schweitsi jutu järele j...,NaN,jutud,saksa,šveitsi,NaN,...,1897.0,NaN,NaN,Tõlkearvustus,Täiskasvanute ilukirjandus,NaN,"28. okt., nr 43, lk 961",NaN,<p>Olevik</p>,NaN
2,"Zunk, Paul","Mällow, J.",NaN,NaN,Miss Portsia,teadmata,jutud,teadmata,teadmata,NaN,...,1894.0,NaN,NaN,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,NaN,nr 21-22,NaN,<p>Eesti Postimees : Lisaleht</p>,Väljaandes: Paul Zunk'i uudisjutukese järele J...
3,"Žukovski, Vassili 1783-1852","Leppik, Jaan 1861-1943",NaN,NaN,Kodu,NaN,luuletused,vene,vene,F. Feldt,...,1889.0,39 lk. 13 cm,NaN,Kogumikus ilmunud tõlge,Täiskasvanute ilukirjandus,NaN,NaN,<p>Ilmunud kogumikus: Eestistatud laulud 2.</p>,NaN,Väljaandes: Wene keelest [autor märkimata]; Ee...
4,"Žukovski, Vassili 1783-1852",G. Kewade,NaN,NaN,Kolm õde,teadmata,jutud,saksa,vene,Laakmann,...,1886.0,NaN,NaN,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,NaN,"31. dets., nr. 52, lk. 409",NaN,<p>Meelelahutaja</p>,Väljaandes: Shukowsky järele: G. Kewade
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123752,"Adamov, Arkadi 1920-1991","Jürna, Juhan-Kaspar 1931-1982",NaN,NaN,Peremeheta kohver,Чемодан без хозяина,jutustused,vene,vene,NaN,...,19721973.0,NaN,NaN,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,NaN,"30. nov.; 1.-3., 5., 7.-8., 10., 12., 13.-17.,...",NaN,<p>Noorte Hääl</p>,NaN
123753,"Abramov, Sergei 1944-2024","Maksing, Meta",NaN,NaN,Nõiatulp : ulmejutt,teadmata,jutud,vene,vene,NaN,...,19801981.0,NaN,NaN,Perioodikas ilmunud tõlge,Laste- ja noortekirjandus,NaN,"12., 16., 26. juuli; 2., 6., 9., 16., 23., 27....",NaN,<p>Säde</p>,NaN
123754,A. K.,"Murakin, Ants 1892-1975",NaN,NaN,Zsigmond Móricz: Mudakuld. Romaan. Ungari keel...,Sárarany,romaanid,ungari,ungari,NaN,...,12929.0,NaN,Looduse kroonine romaan nr 1,Tõlkearvustus,Täiskasvanute ilukirjandus,NaN,"3. veebr., nr 33, lk 8",NaN,<p>Päevaleht</p>,NaN
123755,NaN,"Neumann, Mihkel 1866-1941",NaN,NaN,Irmgard : hästi põnew romaan,teadmata,romaanid,teadmata/undetermined,teadmata/undetermined,teadmata,...,19231924.0,23 annet (733 lk.),NaN,Tõlkeraamat,Täiskasvanute ilukirjandus,NaN,NaN,NaN,NaN,Kaanel pealkirja täiendandmed: Ülipõnew romaan...


In [167]:
data.author.unique()

<StringArray>
[                                  'π',                          'Zunk, Paul',
         'Žukovski, Vassili 1783-1852',                        'Žukov, V. V.',
         'Zshokke, Heinrich 1771-1848', 'Zschokke, Heinrich Daniel 1771-1848',
               'Zola, Émile 1840-1902',     'Zlatovratski, Nikolai 1845-1911',
      'Zitelmann, Katharina 1844-1926',           'Ziethe, Wilhelm 1824-1901',
 ...
              'Ivanov, Juri 1928-1994',        'Horan, James David 1914-1981',
         'Golovtšenko, Ivan 1918-1992',    'Dumas, Alexandre vanem 1802-1870',
                          'Brühl, Al.',                        'Brinker, Jan',
            'Bogurad, Jerzy 1852-1928',                     'Azarov, Aleksei',
               'Kudravtsev, Vladislav',           'Avdejenko, Juri 1933-1982']
Length: 17890, dtype: str

In [169]:
# Created with assistance from Mistral Le Chat
def split_author(author):
    # Find the first digit's index
    match = re.search(r'\d', author)
    if not match:
        return pd.Series({'name': author, 'birth_year': None, 'death_year': None})
    idx = match.start()
    name = author[:idx].strip()
    years = author[idx:].strip()
    birth, *death = years.split('-')
    death_year = death[0] if death else None
    return pd.Series({'name': name, 'birth_year': birth, 'death_year': death_year})

# Apply the function
data['author'].apply(split_author)

TypeError: expected string or buffer

In [153]:
regex1 = re.pattern(r"")
temp = data['author'].str.extract(pattern)

In [166]:
temp[1].unique()

<StringArray>
[   nan, '1771', '1840', '1845', '1844', '1824', '1841', '1868', '1836',
 '1862',
 ...
 '1996', '1717', '1966', '1964', '1634', '1967', '1394', '1968', '1343',
 '1961']
Length: 360, dtype: str